# 03 — Condition-informed FMEA decision support

This notebook audits the separation between a static engineering FMEA scenario and the finalized FD001 condition evidence. It reads generated artifacts and exercises transparent decision rules; it does not retrain a model or redesign the locked analysis.

C-MAPSS provides an engine-level simulated degradation trajectory and end-of-life timing. It does **not** provide row-level labels for the illustrative FMEA failure modes. The scenario table is therefore explicitly `ENGINEERING_SCENARIO_ASSUMPTION`, and its ratings require domain review.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd


def find_development_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src').is_dir() and (candidate / 'requirements.txt').is_file():
            return candidate
    raise RuntimeError('Open this notebook from inside the repository tree.')


DEV_ROOT = find_development_root()
OUTPUT_ROOT = DEV_ROOT.parent / '05_OUTPUTS'
TABLE_DIR = OUTPUT_ROOT / 'tables'
SUMMARY_PATH = OUTPUT_ROOT / 'run_summary.json'
sys.path.insert(0, str(DEV_ROOT))

required_outputs = [
    TABLE_DIR / 'static_fmea.csv',
    TABLE_DIR / 'condition_informed_fmea.csv',
    TABLE_DIR / 'static_vs_condition_comparison.csv',
    TABLE_DIR / 'per_asset_event_metrics.csv',
    TABLE_DIR / 'static_policy_event_metrics.csv',
    SUMMARY_PATH,
]
missing = [str(path) for path in required_outputs if not path.is_file()]
if missing:
    raise FileNotFoundError('Run `python run_analysis.py` first. Missing:\n' + '\n'.join(missing))

OUTPUT_ROOT

## Layer 1 — frozen engineering context

`Static_RPN = Severity x Occurrence x Detection` is retained as a conventional secondary prioritization aid. It is not a predictive target, probability, calibrated quantity, or stand-alone engineering decision. Equal products can conceal different rating patterns, and the ordinal multiplication has known limitations.

In [ ]:
from src.fmea import SCENARIO_BASIS, validate_static_fmea

static_fmea = pd.read_csv(TABLE_DIR / 'static_fmea.csv')
validate_static_fmea(static_fmea)
assert static_fmea['Scenario_Basis'].eq(SCENARIO_BASIS).all()
assert not any('dynamic_rpn' in name.lower() for name in static_fmea.columns)

static_fmea

## Layer 2 — condition observation

The selected model estimates one engine-level probability of entering the 30-cycle simulated end-of-life window. Fixed bands translate that probability into a reviewable observation level: `LOW < 0.20`, `MODERATE >= 0.20`, `HIGH >= 0.50`, and `CRITICAL >= 0.80`. These `PORTFOLIO_SCENARIO_ASSUMPTION` bands are distinct from the development-selected event-warning threshold; they are not industrial or IEC/ISO limits.

Because FD001 does not identify which illustrative failure-mode row is active, applying the same engine-level evidence across scenario rows is a conservative demonstration—not a mode-specific diagnosis.

In [ ]:
from src.decision_support import condition_alert_level

band_audit = pd.DataFrame(
    {'condition_probability': [0.00, 0.19, 0.20, 0.49, 0.50, 0.79, 0.80, 1.00]}
)
band_audit['condition_alert'] = band_audit['condition_probability'].map(condition_alert_level)
band_audit

## Decision-support layer — urgency changes, ratings do not

The condition-informed table may escalate inspection or engineering-review timing when the condition alert is elevated. It must preserve Severity, Occurrence, Detection, and `Static_RPN` exactly. There is no multiplied, blended, or hidden dynamic score.

In [ ]:
condition_fmea = pd.read_csv(TABLE_DIR / 'condition_informed_fmea.csv')
protected = ['Severity', 'Occurrence', 'Detection', 'Static_RPN']
audit = static_fmea[['Failure_Mode', *protected]].merge(
    condition_fmea[['Failure_Mode', *protected]],
    on='Failure_Mode',
    how='inner',
    validate='one_to_one',
    suffixes=('_static', '_condition'),
)
assert len(audit) == len(static_fmea) == len(condition_fmea)
for field in protected:
    left = pd.to_numeric(audit[f'{field}_static'], errors='raise').to_numpy()
    right = pd.to_numeric(audit[f'{field}_condition'], errors='raise').to_numpy()
    assert np.array_equal(left, right), f'{field} was mutated'
assert not any('dynamic_rpn' in name.lower() for name in condition_fmea.columns)

decision_columns = [
    name for name in (
        'Failure_Mode', 'Severity', 'Static_RPN', 'Static_Priority',
        'Condition_Probability', 'Condition_Alert', 'Recommended_Urgency',
        'Condition_Escalated', 'Escalation_Reason',
    ) if name in condition_fmea.columns
]
condition_fmea[decision_columns]

## Static versus condition-informed policy

This is a `SCENARIO_POLICY_ILLUSTRATION`, not "ML versus FMEA." A frozen age/inspection proxy is nearly matched only on development alert-state rows, then both policies are applied unchanged to the official test. Inspected assets, episodes, maintenance hours, cost, asset-level workload, and resource constraints are not matched. Coverage uses only 25 eligible test engines (16 warned; Wilson 95% interval approximately 44.5%–79.8%). The comparison does not establish operational superiority, cost, avoided downtime, failure prevention, or safety benefit.

In [ ]:
policy_comparison = pd.read_csv(TABLE_DIR / 'static_vs_condition_comparison.csv')
condition_events = pd.read_csv(TABLE_DIR / 'per_asset_event_metrics.csv')
static_events = pd.read_csv(TABLE_DIR / 'static_policy_event_metrics.csv')

display(policy_comparison)
display(
    pd.DataFrame(
        {
            'artifact': ['condition_policy_assets', 'static_policy_assets'],
            'rows': [len(condition_events), len(static_events)],
        }
    )
)

## Representative asset and claim boundary

The example asset is selected after model finalization by a deterministic rule: among correctly warned eligible assets, choose the one whose lead time is closest to the median, with asset ID breaking ties. This avoids best-case cherry-picking. The run summary below is the source of record.

In [ ]:
with SUMMARY_PATH.open('r', encoding='utf-8') as handle:
    run_summary = json.load(handle)

summary_view = pd.json_normalize(run_summary, sep='.').T.rename(columns={0: 'value'})
representative_view = summary_view.loc[
    [index for index in summary_view.index if 'representative' in index.lower()]
]
representative_view if not representative_view.empty else summary_view

## What condition information changes—and what it cannot change

Condition evidence can change urgency, monitoring intensity, and the timing of human review for a monitored asset. It does not change the engineering meaning of Severity, validate the illustrative failure-mode mapping, diagnose a specific physical mode, authorize maintenance automatically, or prove improved production or safety outcomes.